# Pipeline Validation: Loop Unrolling Impact

This notebook validates our end-to-end pipeline for measuring loop unrolling impact.

## Goals
1. Compile `simple_loop.c` to LLVM IR
2. Extract loop features from the IR
3. Measure performance with and without loop unrolling
4. Verify that we can detect meaningful performance differences

In [ ]:
import sys
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from compile_and_measure import BenchmarkRunner
from parse_llvm_ir import extract_features_from_ir

# Setup plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Compile to LLVM IR

In [ ]:
# Path to our test program
source_file = project_root / 'benchmarks' / 'simple_loop.c'

# Display the source code
print("Source code:")
print("=" * 60)
print(source_file.read_text())
print("=" * 60)

In [ ]:
# Initialize benchmark runner
runner = BenchmarkRunner(opt_level="O3")

# Compile to LLVM IR (unoptimized to see loop structure clearly)
ir_file = runner.compile_to_llvm_ir(source_file, opt_level="0")
print(f"Generated LLVM IR: {ir_file}")

## 2. Inspect LLVM IR

In [ ]:
# Display first 100 lines of IR to understand structure
ir_content = ir_file.read_text()
lines = ir_content.split('\n')

print("LLVM IR (first 100 lines):")
print("=" * 60)
for i, line in enumerate(lines[:100], 1):
    print(f"{i:3d}: {line}")
print("=" * 60)
print(f"Total lines: {len(lines)}")

## 3. Extract Loop Features

In [ ]:
# Extract features from the IR
features = extract_features_from_ir(ir_file)

if features:
    df_features = pd.DataFrame(features)
    print(f"\nExtracted features for {len(df_features)} loop(s):\n")
    display(df_features.T)  # Transpose for better readability
else:
    print("Warning: No loops detected. This might be due to aggressive optimization.")
    print("Try compiling with -O0 instead.")

## 4. Benchmark Performance Impact

In [ ]:
# Run benchmark
results = runner.benchmark_unrolling_impact(
    source_file,
    num_runs=20,
    warmup_runs=5
)

In [ ]:
# Visualize results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of execution times
categories = ['With Unrolling', 'Without Unrolling']
times = [
    results['with_unroll']['mean'] * 1000,
    results['without_unroll']['mean'] * 1000
]
errors = [
    results['with_unroll']['std'] * 1000,
    results['without_unroll']['std'] * 1000
]

ax1.bar(categories, times, yerr=errors, capsize=5, color=['#2ecc71', '#e74c3c'], alpha=0.8)
ax1.set_ylabel('Execution Time (ms)')
ax1.set_title('Loop Unrolling Performance Comparison')
ax1.grid(axis='y', alpha=0.3)

# Speedup indicator
speedup = results['speedup']
color = '#2ecc71' if speedup > 1.0 else '#e74c3c'
ax2.barh(['Speedup'], [speedup], color=color, alpha=0.8)
ax2.axvline(1.0, color='gray', linestyle='--', linewidth=1, label='Baseline (1.0x)')
ax2.axvline(1.05, color='orange', linestyle=':', linewidth=1, label='Threshold (1.05x)')
ax2.set_xlabel('Speedup Factor')
ax2.set_title(f'Speedup: {speedup:.3f}x')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary
print(f"\n{'Summary':^60}")
print("=" * 60)
print(f"  Speedup from unrolling: {speedup:.3f}x")
print(f"  Classification: {'BENEFICIAL' if results['beneficial'] else 'NOT BENEFICIAL'}")
print(f"  Time saved: {(results['without_unroll']['mean'] - results['with_unroll']['mean']) * 1000:.3f} ms")
print("=" * 60)

## 5. Validation Summary

### Pipeline Components Validated
- ✅ C → LLVM IR compilation
- ✅ Loop feature extraction
- ✅ Binary compilation with/without unrolling
- ✅ Performance measurement
- ✅ Speedup calculation and labeling

### Next Steps
1. Create more diverse benchmark programs
2. Build a dataset with multiple loop patterns
3. Train initial ML model (Logistic Regression / Decision Tree)
4. Compare ML predictions against LLVM's heuristics

In [ ]:
# Save results for reference
import json

output_dir = project_root / 'data' / 'raw'
output_dir.mkdir(parents=True, exist_ok=True)

# Save benchmark results
with open(output_dir / 'simple_loop_benchmark.json', 'w') as f:
    json.dump(results, f, indent=2)

# Save features
if features:
    df_features.to_csv(output_dir / 'simple_loop_features.csv', index=False)

print(f"\nResults saved to: {output_dir}")